In [1]:
import pandas as pd

proteins = pd.read_json("protein.json", orient='index')
proteins.reset_index(inplace=True)

coded_gpcr = pd.read_csv("coded_gpcr_list.csv")
coded_gpcr.drop(columns=['Unnamed: 0'], inplace=True)
#gpcr_proteins = pd.merge(proteins, coded_gpcr, left_on='gpcr_name', right_on='entry_name')

proteins.tail(n=10)

,index,uniprot_id,history_ids,gene_name,gene_synonyms,protein_synonyms,refseq_ids,pubmed_ids,seq_ranges,seq_annotations,pdb_ids,gene_ids,pfam_ids,gpcr_class,entry_name,full_name,species,taxon_id,sequence,sequence_length
3295,Q9TU95,Q9TU95,[],[OR1D5],[],[Olfactory receptor 1D5],[],[pubmed:10512677],[],[],[],[],[PF13853],-,OR1D5_PANPA,OR1D5_PANPA,Pan paniscus,9597,MDGDNQSENSQFLLLGISESPEQQQILFWMFLSMYLVTVLGNVLII...,312 AA
3296,Q9TU97,Q9TU97,[],[OR3A2],[],[Olfactory receptor 3A2],[],[pubmed:10512677],[],[],[],[],[PF13853],-,OR3A2_PANTR,OR3A2_PANTR,Pan troglodytes,9598,MEPEAGTNRTAVAEFILLGLVQTEEMQPVVFVLFLFAYLVTIGGNL...,315 AA
3297,Q9TU99,Q9TU99,[],[OR1G1],[],[Olfactory receptor 1G1],[NP_001009085.1],[pubmed:10512677],[],[],[],[454426],[PF13853],-,OR1G1_PANTR,OR1G1_PANTR,Pan troglodytes,9598,MEGKNLTSISEFFLLGFSEQLEEQKALFGSFLFMYLVMVAGNLLII...,313 AA
3298,Q9TUA0,Q9TUA0,[],[OR3A3],[],[Olfactory receptor 3A3],[],[pubmed:10512677],[],[],[],[],[PF13853],-,OR3A3_PANTR,OR3A3_PANTR,Pan troglodytes,9598,MESEAGTNRTAVAEFMLLGLVQTEEMQSVIFVLLLFAYLVTTGGNL...,315 AA
3299,Q9TUA1,Q9TUA1,[],[OR1E5],[],[Olfactory receptor 1E5],[NP_001009164.1],[pubmed:10512677],[],[],[],[494123],[PF13853],-,OR1E5_PANTR,OR1E5_PANTR,Pan troglodytes,9598,MMGQNQTSISDFLLLGLPIQPEQQNLCYALFLAMYLTTLLGNLLII...,314 AA
3300,Q9TUA4,Q9TUA4,[],[OR3A1],[],[Olfactory receptor 3A1],[],[pubmed:10512677],[],[],[],[],[PF13853],-,OR3A1_PANTR,OR3A1_PANTR,Pan troglodytes,9598,MQPESGANGTVIAEFILLGLLEAPGLQPVVFVLFLFAYLVTVGGNL...,315 AA
3301,Q9TUA6,Q9TUA6,[],[OR1D5],[],[Olfactory receptor 1D5],[NP_001009113.1],[pubmed:10512677],[],[],[],[468144],[PF13853],-,OR1D5_PANTR,OR1D5_PANTR,Pan troglodytes,9598,MDGDNQSENSQFLLLGISESPEQQQILFWMFLSMYLVTVLGNVLII...,312 AA
3302,Q9TUA7,Q9TUA7,[],[OR1A1],[],[Olfactory receptor 1A1],"[NP_001009115.1, XP_054525812.1, XP_054525813.1]",[pubmed:10512677],[],[],[],[468149],[PF13853],-,OR1A1_PANTR,OR1A1_PANTR,Pan troglodytes,9598,MRENNQSSTLEFILLGVTGQQEQEDFFYILFLFIYPITLIGNLLIV...,309 AA
3303,P83854,P83854,[],[],[],[Putative G-protein coupled receptor],[],[],[],[],[],[],[],-,GPCR_MOUSE,GPCR_MOUSE Putative G-protein coupled receptor...,Mus musculus,10090,GPQQENMMEE,10 AA
3304,Q09964,Q09964,[Q2PJA8],[B0244.4],[],[Putative G-protein coupled receptor B0244.4],[NP_001254931.1],[pubmed:9851916],[],[],[],[181875],[],-,YS94_CAEEL,YS94_CAEEL Putative G-protein coupled receptor...,Caenorhabditis elegans,6239,MANIFENCSYHSPYEPYYLNCTNTTNQCTLVQDVETIESIIFWIDF...,309 AA


In [2]:
import re
import difflib

# 1) Build a normalized name index from protein.json names

def normalize_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    name = name.lower().strip()
    name = name.replace("β", "beta").replace("α", "alpha")
    name = re.sub(r"[^a-z0-9\s-]", " ", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def strip_leading_token_from_full_name(full_name: str) -> str:
    # e.g. "GPR4_XENTR G-protein coupled receptor 4" -> "G-protein coupled receptor 4"
    if not isinstance(full_name, str):
        return ""
    parts = full_name.split(" ", 1)
    if len(parts) == 2 and "_" in parts[0]:
        return parts[1]
    return full_name


name_to_protein_rows = {}
all_norm_names = set()

# essentially builds a lookup structure of normalized names to protein rows
for _, prow in proteins.iterrows():
    candidates = []

    for s in prow.get("protein_synonyms", []):
        if isinstance(s, str) and s.strip():
            candidates.append(s)

    full_name = prow.get("full_name", "")
    if isinstance(full_name, str) and full_name.strip():
        candidates.append(full_name)
        stripped = strip_leading_token_from_full_name(full_name)
        if stripped and stripped != full_name:
            candidates.append(stripped)

    for cname in candidates:
        key = normalize_name(cname)
        if not key:
            continue
        all_norm_names.add(key)
        name_to_protein_rows.setdefault(key, []).append(prow)


# 2) For each coded GPCR name, find matching proteins
#    - exact normalized match first
#    - fallback to fuzzy match (subtypes/species/isoforms are allowed to be approximate)

def match_gpcr_name(gpcr_name: str, fuzzy_cutoff: float = 1):
    key = normalize_name(gpcr_name)
    if key in name_to_protein_rows:
        return key, name_to_protein_rows[key], "exact"

    close = difflib.get_close_matches(key, all_norm_names, n=1, cutoff=fuzzy_cutoff)
    if close:
        best = close[0]
        return best, name_to_protein_rows[best], "fuzzy"

    return None, [], "none"


# 3) Build class-level sequence table
#    Keeps all matched proteins; duplicates by uniprot_id removed per class

unique_classes = sorted(coded_gpcr["gpcr_binding_encoded"].dropna().unique())
class_rows = []

for class_id in unique_classes:
    class_names = sorted(
        set(coded_gpcr.loc[coded_gpcr["gpcr_binding_encoded"] == class_id, "gpcr_name"].dropna())
    )

    seen_uniprot = set()
    for gpcr_name in class_names:
        matched_key, matched_proteins, match_type = match_gpcr_name(gpcr_name)

        for prow in matched_proteins:
            uid = prow.get("uniprot_id")
            if not isinstance(uid, str) or not uid or uid in seen_uniprot:
                continue

            seq = prow.get("sequence")
            if not isinstance(seq, str) or not seq.strip():
                continue

            seen_uniprot.add(uid)
            class_rows.append(
                {
                    "gpcr_binding_encoded": int(class_id),
                    "gpcr_name_query": gpcr_name,
                    "match_type": match_type,
                    "matched_name_key": matched_key,
                    "uniprot_id": uid,
                    "entry_name": prow.get("entry_name"),
                    "species": prow.get("species"),
                    "sequence": seq,
                }
            )


class_sequences_df = pd.DataFrame(class_rows)

n_classes_with_sequences = class_sequences_df["gpcr_binding_encoded"].nunique()
print(f"Classes requested: {len(unique_classes)}")
print(f"Classes with >=1 sequence: {n_classes_with_sequences}")
print(f"Total extracted sequences: {len(class_sequences_df)}")
print(f"Total extracted sequences: {len(class_sequences_df['sequence'])}")
print("--------------------------------")
print("Match type distribution:")
print(class_sequences_df["match_type"].value_counts(normalize=True))

class_sequences_df.to_csv("gpcr_class_sequences_from_json.csv", index=False)

class_sequences_df.head(20)

Classes requested: 88
Classes with >=1 sequence: 81
Total extracted sequences: 1113
Total extracted sequences: 1113
--------------------------------
Match type distribution:
match_type
exact    1.0
Name: proportion, dtype: float64


,gpcr_binding_encoded,gpcr_name_query,match_type,matched_name_key,uniprot_id,entry_name,species,sequence
0,0,2-oxoglutarate receptor 1,exact,2-oxoglutarate receptor 1,Q96P68,OXGR1_HUMAN,Homo sapiens,MNEPLDYLANASDFPDYAAAFGNCTDENIPLKMHYLPVIYGIIFLV...
1,0,2-oxoglutarate receptor 1,exact,2-oxoglutarate receptor 1,Q6IYF8,OXGR1_MOUSE,Mus musculus,MIEPLDSPASDSDFLDYPSALGNCTDEQISFKMQYLPVIYSIIFLV...
2,0,2-oxoglutarate receptor 1,exact,2-oxoglutarate receptor 1,Q6Y1R5,OXGR1_RAT,Rattus norvegicus,MIETLDSPANDSDFLDYITALENCTDEQISFKMQYLPVIYSIIFLV...
3,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,P08908,5HT1A_HUMAN,Homo sapiens,MDVLSPGQGNNTTSPPAPFETGGNTTGISDVTVSYQVITSLLLGTL...
4,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,P19327,5HT1A_RAT,Rattus norvegicus,MDVFSFGQGNNTTASQEPFGTGGNVTSISDVTFSYQVITSLLLGTL...
5,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,Q0EAB6,5HT1A_HORSE,Equus caballus,MDVLGPGQGNNTTSSEGPFGTRANATGISDVTFSYQVITSLLLGTL...
6,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,Q64264,5HT1A_MOUSE,Mus musculus,MDMFSLGQGNNTTTSLEPFGTGGNDTGLSNVTFSYQVITSLLLGTL...
7,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,Q6XXX9,5HT1A_CANLF,Canis lupus familiaris,MEGLSPRQGNNTTSSEGPFGTLGNATGISDVTFSYQVITSLLLGTL...
8,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,Q98998,5HT1A_XENLA,Xenopus laevis,MDASNNTTSWNILQRGRMGPSWRRCPVSYQIIASLFLGRSFSAGIF...
9,1,5-hydroxytryptamine receptor 1A,exact,5-hydroxytryptamine receptor 1a,Q9N297,5HT1A_GORGO,Gorilla gorilla gorilla,MDVLSPGQGNNTTSPPAPFETGGNTTGISDVTFSYQVITSLLLGTL...
